# LinguoMT — Central Experiment Runner

Runs all four pipelines for a selected paper mode. Individual experiments
can be disabled by commenting them out in the `EXPERIMENTS` list (Step 3).

| # | Experiment | Model | Dataset | Languages |
|---|---|---|---|---|
| 1 | AfricanCeltic × SeamlessM4T-v2 | `facebook/seamless-m4t-v2-large` | African-Celtic | Igbo · Yoruba |
| 2 | AfricanCeltic × Whisper+NLLB | `whisper-large-v3` + `nllb-200-distilled-600M` | African-Celtic | Yoruba · Hausa |
| 3 | FLEURS × SeamlessM4T-v2 | `facebook/seamless-m4t-v2-large` | FLEURS | Igbo · Yoruba · Swahili |
| 4 | FLEURS × Whisper+NLLB | `whisper-large-v3` + `nllb-200-distilled-600M` | FLEURS | Yoruba · Hausa · Swahili |

| Mode | Per experiment | Total |
|------|---------------|-------|
| DEBUG | ~10 min | ~40 min |
| FULL  | ~30–60 min | ~2–4 hours |

> **Before running:** Runtime → Change runtime type → **T4 GPU** (or A100).

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone / Update Repository

In [ ]:
import os, subprocess

REPO_DIR = "/content/LinguoMT-AfricaS2T"
REPO_URL = "https://github.com/prsisda/LinguoMT-AfricaS2T.git"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print("Repo reset to origin/main")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 3 — Configure Paper Mode & Select Experiments

In [ ]:
import re, pathlib

# ── SELECT PAPER (uncomment the one to run) ───────────────────────────
PAPER_MODE = "benchmark"    # Paper 1 — zero-shot baselines          ← ACTIVE
# PAPER_MODE = "adaptation" # Paper 2 — fine-tuning comparison
# PAPER_MODE = "audio"      # Paper 3 — audio strategy analysis
# PAPER_MODE = "cascade"    # Paper 4 — cascade vs end-to-end
# PAPER_MODE = "transfer"   # Paper 5 — cross-lingual transfer
# ─────────────────────────────────────────────────────────────────────

# ── SELECT MODE ───────────────────────────────────────────────────────
DEBUG_MODE = True   # True ≈ 40 min total | False ≈ 2–4 hours total
# ─────────────────────────────────────────────────────────────────────

# ── SELECT EXPERIMENTS (comment out any you want to skip) ─────────────
EXPERIMENTS = [
    "AfricanCeltic__SeamlessM4Tv2",
    "AfricanCeltic__WhisperNLLB",
    "FLEURS__SeamlessM4Tv2",
    "FLEURS__WhisperNLLB",
]
# ─────────────────────────────────────────────────────────────────────

SCRIPTS = {exp: pathlib.Path(f"{exp}/notebooks/run_experiment.py") for exp in EXPERIMENTS}

mode_str = "True" if DEBUG_MODE else "False"
for name, sp in SCRIPTS.items():
    src = sp.read_text()
    src = re.sub(r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)", rf"\g<1>{mode_str}", src)
    src = re.sub(r'(?m)^(PAPER_MODE\s*=\s*)["\']\w+["\']', rf'\g<1>"{PAPER_MODE}"', src)
    sp.write_text(src)
    print(f"Configured: {name}")

print(f"\nPaper : {PAPER_MODE}")
print(f"Mode  : {'DEBUG  (fast test)' if DEBUG_MODE else 'FULL   (paper run)'}")
print(f"Runs  : {len(SCRIPTS)} experiment(s)")

## Step 4 — Pull Latest & Run All Experiments

In [ ]:
import subprocess, re, pathlib

# Pull latest fixes before running
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("Repository updated to latest.")

# Re-apply config after pull
_mode_str = "True" if DEBUG_MODE else "False"
for name, sp in SCRIPTS.items():
    src = sp.read_text()
    src = re.sub(r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)", rf"\g<1>{_mode_str}", src)
    src = re.sub(r'(?m)^(PAPER_MODE\s*=\s*)["\']\w+["\']', rf'\g<1>"{PAPER_MODE}"', src)
    sp.write_text(src)
print(f"Config re-applied — Paper: {PAPER_MODE} | Mode: {'DEBUG' if DEBUG_MODE else 'FULL'}")

In [ ]:
import sys

for name, sp in SCRIPTS.items():
    print(f"\n{'='*64}")
    print(f"  Running : {name}")
    print(f"  Paper   : {PAPER_MODE}  |  Mode: {'DEBUG' if DEBUG_MODE else 'FULL'}")
    print(f"{'='*64}\n")
    script = str(sp)
    !python "$script"

print("\nAll selected experiments finished.")

---
## Step 5 — Consolidate & Save Outputs

Merges metrics from all runs into a single `summary.md` and master CSVs.  
Individual experiment output folders are **not modified**.

In [ ]:
import json as _json, shutil
import pandas as pd
from pathlib import Path
from datetime import datetime

out_root         = Path("/content/outputs")
consolidated_dir = out_root / f"consolidated_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
consolidated_dir.mkdir(parents=True, exist_ok=True)

all_text, all_asr = [], []
for run_dir in sorted(out_root.glob("*/")):
    if "consolidated" in run_dir.name:
        continue
    exp_label = run_dir.name
    cfg_path  = run_dir / "config.json"
    if cfg_path.exists():
        exp_label = _json.loads(cfg_path.read_text()).get("experiment_family", run_dir.name)
    for fname, store in [("text_metrics.csv", all_text), ("asr_metrics.csv", all_asr)]:
        fpath = run_dir / "metrics" / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            if "experiment" not in df.columns:
                df.insert(0, "experiment", exp_label)
            store.append(df)

summary = [f"# LinguoMT Results — {PAPER_MODE}\n\n"]
if all_text:
    df_text = pd.concat(all_text, ignore_index=True)
    df_text.to_csv(consolidated_dir / "text_metrics_all.csv", index=False)
    summary += ["## Text Translation\n\n", df_text.to_markdown(index=False), "\n\n"]
    print("=== Text Metrics ==="); print(df_text.to_string(index=False))
if all_asr:
    df_asr = pd.concat(all_asr, ignore_index=True)
    df_asr.to_csv(consolidated_dir / "asr_metrics_all.csv", index=False)
    summary += ["## ASR\n\n", df_asr.to_markdown(index=False), "\n"]
    print("\n=== ASR Metrics ==="); print(df_asr.to_string(index=False))

(consolidated_dir / "summary.md").write_text("".join(summary))
print(f"\nConsolidated: {consolidated_dir}")

if Path("/content/drive/MyDrive").exists():
    drive_dest = Path("/content/drive/MyDrive/LinguoMT-AfricaS2T") / consolidated_dir.name
    drive_dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(consolidated_dir), str(drive_dest))
    print(f"Backed up to Drive: {drive_dest}")